## Persistent Landing - Delta Lake

The aim of this notebook is to convert all structure data we have into parquet files, in our case, it would be CSV files.

**Importing Useful Libraries**

In [1]:
import os
import boto3
import duckdb
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from deltalake import DeltaTable, write_deltalake
import polars as pl

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
MINIO_ROLE = "writer"
if MINIO_ROLE == "admin":
    access_key = os.getenv("MINIO_ACCESS_KEY")
    secret_key = os.getenv("MINIO_SECRET_KEY")
else:
    role_prefix = MINIO_ROLE.upper()
    access_key = os.getenv(f"MINIO_{role_prefix}_ACCESS_KEY")
    secret_key = os.getenv(f"MINIO_{role_prefix}_SECRET_KEY")
if not endpoint or not access_key or not secret_key:
    raise RuntimeError(f"Missing MinIO {MINIO_ROLE} credentials in environment")


In [2]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)

In [3]:
# Connect to DuckDB and configure S3 secret for MinIO
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET secret (
    TYPE s3,
    PROVIDER config,
    ENDPOINT '{endpoint.replace("http://", "").replace("https://", "")}',
    KEY_ID '{access_key}',
    SECRET '{secret_key}',
    URL_STYLE 'path',
    USE_SSL false
);
""")

This function performs **zero-schema data integrity verification** between a source CSV file and a Delta table.

It is designed to validate:

- Row consistency
- Column consistency
- Content-level equality (order-agnostic)

The verification does **not rely on schema definitions**, making it suitable for flexible or evolving datasets.


In [4]:
def verify_data_integrity(original_csv, delta_table_path, db_connection):
    """
    Zero-Schema Data Integrity Verification for Notebooks.
    """
    # Initialize connection
    con = db_connection

    print("--- Starting Verification ---")

    # Load Delta extension
    try:
        con.execute("INSTALL delta; LOAD delta;")
    except Exception:
        pass

    try:
        # STEP 1: Structural Validation (Row/Col counts)
        structure_sql = f"""
            SELECT
                (SELECT count(*) FROM read_csv_auto('{original_csv}')) as csv_rows,
                (SELECT count(*) FROM delta_scan('{delta_table_path}')) as delta_rows,
                (SELECT count(*) FROM (DESCRIBE SELECT * FROM read_csv_auto('{original_csv}'))) as csv_cols,
                (SELECT count(*) FROM (DESCRIBE SELECT * FROM delta_scan('{delta_table_path}'))) as delta_cols
        """
        csv_rows, delta_rows, csv_cols, delta_cols = con.execute(structure_sql).fetchone()

        if csv_rows != delta_rows or csv_cols != delta_cols:
            print(f"❌ Mismatch! CSV: {csv_rows}r/{csv_cols}c, Delta: {delta_rows}r/{delta_cols}c")
            return False

        # STEP 2: Content Inspection (Order-agnostic fingerprinting)
        # Cast to VARCHAR to avoid type-storage conflicts
        fingerprint_sql = f"""
            WITH csv_sig AS (
                SELECT bit_xor(hash(coalesce(columns(*)::VARCHAR, 'NULL'))) as sign
                FROM read_csv_auto('{original_csv}')
            ),
            delta_sig AS (
                SELECT bit_xor(hash(coalesce(columns(*)::VARCHAR, 'NULL'))) as sign
                FROM delta_scan('{delta_table_path}')
            )
            SELECT csv_sig.sign == delta_sig.sign FROM csv_sig, delta_sig
        """

        is_identical = con.execute(fingerprint_sql).fetchone()[0]

        if not is_identical:
            print("❌ Integrity Failed: Fingerprints do not match.")
            return False

        print(f"✅ Passed: {csv_rows} rows and {csv_cols} columns are identical.")
        return True

    except Exception as e:
        print(f"⚠️ Error during verification: {e}")
        return False

To build a high-performance "**Bronze Layer**" (this is simulated by the sub-bucket `csv-delta-lake`), this pipeline automates the conversion of raw CSV files into versioned **Delta Tables** by leveraging **DuckDB** for high-speed S3 streaming and Parquet conversion, followed by **Polars** to finalize the ACID-compliant Delta format. This transition optimizes the data lake for columnar performance and transactional integrity (via the `_delta_log`), while maintaining a clean environment by automatically purging intermediate Parquet files and providing full compatibility with S3-compatible storage through specialized `storage_options`.

In [5]:
storage_options = {
    "AWS_ACCESS_KEY_ID": access_key,
    "AWS_SECRET_ACCESS_KEY": secret_key,
    "AWS_ENDPOINT_URL": endpoint,
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    "AWS_S3_ADDRESSING_STYLE": "path",
    "AWS_ALLOW_HTTP": "true",
    "region": "us-east-1"
}

def ingest_csv_to_delta(bucket, csv_prefix="persistent-landing/structured/raw/"):
    """
    Unified pipeline: 
    1. DuckDB reads CSV and converts to a temporary Parquet.
    2. Polars/DeltaLake reads that Parquet and writes it as a versioned Delta Table.
    3. Cleans up the temporary Parquet.
    """
    # Initialize DuckDB S3 access
    con.execute("INSTALL httpfs; LOAD httpfs;")
    
    paginator = s3.get_paginator("list_objects_v2")
    delta_base_prefix = "persistent-landing/structured"
    
    for page in paginator.paginate(Bucket=bucket, Prefix=csv_prefix):
        for obj in page.get("Contents", []):
            src_key = obj["Key"]
            
            # Skip directories
            if obj['Size'] == 0 and src_key.endswith("/"):
                continue

            # Extract base name (e.g., 'users' from 'path/to/users.csv')
            file_base = os.path.splitext(os.path.basename(src_key))[0]
            
            # Path Definitions
            s3_csv_path = f"s3://{bucket}/{src_key}"
            temp_parquet_path = f"s3://{bucket}/{delta_base_prefix}/{file_base}_temp.parquet"
            target_delta_folder = f"s3://{bucket}/{delta_base_prefix}/{file_base}/"

            print(f"🚀 Processing: {file_base}...")

            try:
                # --- Phase 1: DuckDB (CSV to Parquet) ---
                con.execute(f"""
                    COPY (SELECT * FROM read_csv_auto('{s3_csv_path}')) 
                    TO '{temp_parquet_path}' (FORMAT PARQUET);
                """)
                print(f"  └─ DuckDB: CSV converted to temporary Parquet.")

                # --- Phase 2: Delta Lake (Parquet to Delta) ---
                df = pl.read_parquet(temp_parquet_path, storage_options=storage_options)
                
                write_deltalake(
                    target_delta_folder,
                    df,
                    mode="overwrite",
                    storage_options=storage_options
                )
                print(f"  └─ Delta: Version 0 created at {target_delta_folder}")

                # --- Phase 3: Cleanup ---
                # Delete the temporary parquet file
                s3.delete_object(Bucket=bucket, Key=f"{delta_base_prefix}/{file_base}_temp.parquet")
                print(f"✅ Successfully finalized {file_base}.")
                if verify_data_integrity(s3_csv_path,target_delta_folder,con):
                    s3.delete_object(Bucket=bucket, Key=f"{csv_prefix}{file_base}.csv")
                    print(f"✅ Successfully deleted csv file {s3_csv_path}.")

                else:
                    raise ValueError("Verification process crashed")


            except Exception as e:
                print(f"❌ Failed to process {file_base}: {e}")

In [6]:
# Convert csv files into parquet files
ingest_csv_to_delta("landing-zone")

🚀 Processing: co2-emission-by-vehicles_1779622021500...
  └─ DuckDB: CSV converted to temporary Parquet.
  └─ Delta: Version 0 created at s3://landing-zone/persistent-landing/structured/co2-emission-by-vehicles_1779622021500/
✅ Successfully finalized co2-emission-by-vehicles_1779622021500.
--- Starting Verification ---
✅ Passed: 7385 rows and 12 columns are identical.
✅ Successfully deleted csv file s3://landing-zone/persistent-landing/structured/raw/co2-emission-by-vehicles_1779622021500.csv.
🚀 Processing: global_warming_dataset_1779622021583...
  └─ DuckDB: CSV converted to temporary Parquet.
  └─ Delta: Version 0 created at s3://landing-zone/persistent-landing/structured/global_warming_dataset_1779622021583/
✅ Successfully finalized global_warming_dataset_1779622021583.
--- Starting Verification ---
✅ Passed: 100000 rows and 26 columns are identical.
✅ Successfully deleted csv file s3://landing-zone/persistent-landing/structured/raw/global_warming_dataset_1779622021583.csv.
🚀 Proces

We can now try querying over these parquet filed.

In [7]:
print("🔎 Reading first 10 rows of co2-emission.parquet via DuckDB:")
table_path = "s3://landing-zone/persistent-landing/structured/global_warming*/part*.parquet"
df_view = con.execute(f"SELECT * FROM read_parquet('{table_path}') LIMIT 10").df()
df_view

🔎 Reading first 10 rows of co2-emission.parquet via DuckDB:


,Country,Year,Temperature_Anomaly,CO2_Emissions,Population,Forest_Area,GDP,Renewable_Energy_Usage,Methane_Emissions,Sea_Level_Rise,...,Waste_Management,Per_Capita_Emissions,Industrial_Activity,Air_Pollution_Index,Biodiversity_Index,Ocean_Acidification,Fossil_Fuel_Usage,Energy_Consumption_Per_Capita,Policy_Score,Average_Temperature
0,Country_103,1913,-1.163537,8.876061e+08,1.627978e+08,54.872178,6.139887e+12,76.710013,8.317626e+06,8.111839,...,82.691409,2.285351,4.060975,150.285539,90.073356,8.025470,39.163860,1480.164332,78.870012,20.825292
1,Country_180,1950,-0.432122,4.497517e+08,4.281359e+08,84.051006,2.601447e+12,68.450021,6.206540e+06,42.025915,...,59.322883,17.411668,85.300604,27.305922,88.289837,8.021719,28.252554,1482.730048,32.600905,28.720587
2,Country_93,2014,0.444954,4.579080e+08,4.926732e+08,72.295357,5.192677e+12,36.725699,1.056885e+06,20.953840,...,94.982931,12.039703,83.804880,216.911429,86.936256,7.647408,61.548382,706.918809,37.671300,15.014084
3,Country_15,2020,-1.171616,5.049503e+08,1.252169e+09,17.259684,8.252128e+12,77.547901,1.986813e+06,45.599595,...,62.064250,2.853957,47.014265,35.869182,44.904331,7.569353,82.423750,2616.238324,86.581725,-1.277086
4,Country_107,1964,-0.564038,6.898891e+08,2.932960e+08,44.438605,8.560746e+12,10.019576,3.313252e+06,7.652150,...,84.431279,19.801173,89.379613,284.263093,8.102916,8.015415,29.964450,4975.683780,20.618406,2.861989
5,Country_72,1926,-1.946218,2.394448e+08,1.441203e+07,6.117781,1.989459e+12,26.163399,8.218906e+06,13.574102,...,36.112468,8.528017,63.807031,172.964007,20.512193,7.874246,67.153067,267.465131,84.672317,37.889804
6,Country_189,1921,-1.362100,1.977445e+08,1.339563e+09,75.554131,9.131238e+12,82.661868,5.160613e+06,1.378949,...,1.237686,7.484364,15.002742,238.288021,67.988814,8.448967,22.046013,3114.698772,72.740043,-7.786539
7,Country_21,1921,1.627624,2.111461e+08,2.410014e+07,25.999981,4.034784e+12,36.773252,5.890986e+06,15.617050,...,90.687488,15.747481,86.194366,150.058727,44.592418,8.436721,58.698935,3519.096247,19.243964,4.206395
8,Country_103,1989,0.541387,7.802026e+08,3.855137e+08,17.797781,6.823672e+12,3.422548,1.313058e+04,36.226933,...,60.895322,10.568785,66.175268,242.089767,93.935792,8.224373,46.630095,2718.348573,89.810124,-5.829195
9,Country_122,1963,0.016534,5.397029e+08,5.508798e+08,8.948443,4.713484e+12,72.875002,5.766624e+06,8.016881,...,77.485505,13.919414,70.265639,153.191500,77.122932,8.190353,52.587813,1745.691747,30.215953,2.473330
